In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import hashlib
import secrets
import time

In [ ]:
# Helper functions

def measure_time(func, *args, **kwargs):
    """Run func(*args) and return (result, elapsed_ms)."""
    t0 = time.perf_counter()
    result = func(*args, **kwargs)
    return result, (time.perf_counter() - t0) * 1000


def H(*args) -> bytes:
    """
    Hash function (random oracle).
    Accepts ints, bytes, or numpy uint8 arrays.
    Returns 32 bytes.
    """
    h = hashlib.sha256()
    for a in args:
        if isinstance(a, int):
            h.update(a.to_bytes(8, 'big'))
        elif isinstance(a, (bytes, bytearray)):
            h.update(a)
        elif isinstance(a, np.ndarray):
            h.update(a.tobytes())   # deterministic: each uint8 element → 1 byte
    return h.digest()


def xor_bytes(a: bytes, b: bytes) -> bytes:
    """XOR two equal-length byte strings."""
    return bytes(x ^ y for x, y in zip(a, b))


def prg(seed: bytes, n_bits: int) -> np.ndarray:
    """
    Pseudorandom Generator — SHA-256 in counter mode.
    Expands `seed` (bytes) into `n_bits` pseudorandom bits (np.uint8 array).
    """
    buf = bytearray()
    ctr = 0
    while len(buf) * 8 < n_bits:
        buf += hashlib.sha256(seed + ctr.to_bytes(4, 'big')).digest()
        ctr += 1
    return np.unpackbits(np.frombuffer(bytes(buf), dtype=np.uint8))[:n_bits].astype(np.uint8)

In [ ]:
# %% [markdown]
# # Secure Multiparty Computation — OT vs OT Extension
# Comparing **Simple Oblivious Transfer** (DH-based) against
# **IKNP OT Extension** (Ishai, Kilian, Nissim, Petrank 2003).

# ── DH group ──────────────────────────────────────────────────────────────
# Using the Mersenne prime 2^61 − 1 for fast demo arithmetic.
# A production implementation would use a 2048-bit safe prime (e.g. RFC 3526).
DH_P = (1 << 61) - 1   # Mersenne prime, 61 bits
DH_G = 2               # generator

In [ ]:
# Simple Oblivious Transfer (OT) implementaiton - Naor-Pinkas DH-based 1-out-of-2 OT ──────────────────

def simple_ot(m0: bytes, m1: bytes, choice: int) -> bytes:
    """
    Naor-Pinkas 1-out-of-2 Oblivious Transfer (DH-based).

    Protocol overview (simulated single-machine):
    ┌──────────────────────────────────────────────────────────┐
    │  Sender knows (m0, m1).                                  │
    │  Receiver knows choice ∈ {0,1}, receives m_{choice}.    │
    │                                                          │
    │  1. Sender:   a ← ℤ_P,  A = g^a                        │
    │  2. Receiver: b ← ℤ_P                                   │
    │               B = g^b        if choice=0                │
    │               B = A·g^b      if choice=1                │
    │               k_recv = A^b                              │
    │  3. Sender:   k0 = B^a,  k1 = (B/A)^a                  │
    │               e0 = m0 ⊕ H(k0),  e1 = m1 ⊕ H(k1)       │
    │  4. Receiver: m_{choice} = e_{choice} ⊕ H(k_recv)      │
    └──────────────────────────────────────────────────────────┘

    Security: computationally secure under the DDH assumption.
    Cost:     O(1) modular exponentiations per OT  ← the expensive part.
    """
    P, G = DH_P, DH_G

    # ── Sender setup ──────────────────────────────────────────────────────
    a = secrets.randbelow(P - 2) + 2
    A = pow(G, a, P)                        # A = g^a  [mod P]

    # ── Receiver encodes choice in public key B ───────────────────────────
    b = secrets.randbelow(P - 2) + 2
    if choice == 0:
        B = pow(G, b, P)                    # B = g^b
    else:
        B = (A * pow(G, b, P)) % P          # B = A · g^b
    k_recv = pow(A, b, P)                   # = A^b = g^(ab)

    # ── Sender builds two encryption keys ─────────────────────────────────
    k0 = pow(B, a, P)                       # k0 = B^a
    A_inv = pow(A, P - 2, P)               # A^{−1} mod P  (Fermat's little theorem)
    k1 = pow((B * A_inv) % P, a, P)        # k1 = (B/A)^a

    # ── Sender encrypts both messages ─────────────────────────────────────
    e0 = xor_bytes(m0, H(k0)[:len(m0)])
    e1 = xor_bytes(m1, H(k1)[:len(m1)])

    # ── Receiver decrypts the chosen message ──────────────────────────────
    pad = H(k_recv)[:len(m0)]
    return xor_bytes(e0 if choice == 0 else e1, pad)


# ── correctness check ─────────────────────────────────────────────────────
assert simple_ot(b"hello_m0__", b"hello_m1__", 0) == b"hello_m0__"
assert simple_ot(b"hello_m0__", b"hello_m1__", 1) == b"hello_m1__"
print("Simple OT  ✓  correctness check passed")


def bench_simple_ot(n: int) -> float:
    """Run n independent Simple OTs. Returns total wall-clock time in ms."""
    m0, m1 = secrets.token_bytes(16), secrets.token_bytes(16)
    t0 = time.perf_counter()
    for i in range(n):
        simple_ot(m0, m1, i % 2)
    return (time.perf_counter() - t0) * 1000


In [ ]:
# OT Extension (OTE) implementaiton

def iknp_ote(sender_m0: list,
             sender_m1: list,
             receiver_choices: np.ndarray,
             k: int = 128) -> list:
    """
    IKNP OT Extension — semi-honest (Ishai-Kilian-Nissim-Petrank, 2003).

    Converts k expensive base OTs into m = len(receiver_choices) cheap OTs.

    Key idea
    --------
    Instead of running m DH-based OTs (cost = m × modexp), we:
      1. Run k base OTs (cost = k × modexp) — the FIXED setup cost.
      2. Use PRG + XOR + hashing to "extend" to m OTs cheaply — cost ≈ m × hash.

    For large m  →  amortized cost per OT → 0  (just hashing).

    Protocol sketch
    ---------------
    Notation: m messages, k security parameter, r ∈ {0,1}^m receiver choices.

    Step 1 — Sender samples s ∈ {0,1}^k  (random correlation string).

    Step 2 — k base OTs  [R is OT-sender, S is OT-receiver with choice s_j]:
        R has (seeds0_j, seeds1_j).  S receives q_j = seeds_{s_j}_j.

    Step 3 — R builds matrix T (k×m) and correction U:
        T[j]  = PRG(seeds0_j)                ← m pseudorandom bits
        U[j]  = PRG(seeds1_j) ⊕ T[j] ⊕ r    ← sent to S

    Step 4 — S builds matrix Q:
        Q[j]  = PRG(q_j) ⊕ s_j · U[j]
              = T[j]          if s_j = 0
              = T[j] ⊕ r      if s_j = 1
        ⟹ Q_i  = T_i ⊕ r_i · s   (i-th row, k bits)

    Step 5 — Transfer m messages (cheap hashing):
        Sender:   e0_i = m0_i ⊕ H(i, Q_i)
                  e1_i = m1_i ⊕ H(i, Q_i ⊕ s)
        Receiver: recv_key = H(i, T_i)
                  if r_i=0: H(T_i) = H(Q_i)        → decrypts e0_i ✓
                  if r_i=1: H(T_i) = H(Q_i ⊕ s)    → decrypts e1_i ✓

    Args
    ----
    sender_m0, sender_m1 : lists of m byte-strings (sender's message pairs).
    receiver_choices     : np.uint8 array of m choice bits.
    k                    : security parameter (number of base OTs, default 128).

    Returns
    -------
    List of m received byte-strings.
    """
    m = len(receiver_choices)
    r = receiver_choices.astype(np.uint8)                # (m,)

    # ── Step 1: Sender samples s ∈ {0,1}^k ───────────────────────────────
    s_raw = secrets.token_bytes((k + 7) // 8)
    s = np.unpackbits(np.frombuffer(s_raw, dtype=np.uint8))[:k].astype(np.uint8)  # (k,)

    # ── Step 2: k base OTs ────────────────────────────────────────────────
    # R provides seed pairs; S picks with s[j].  Only k DH operations here.
    seeds0 = [secrets.token_bytes(16) for _ in range(k)]
    seeds1 = [secrets.token_bytes(16) for _ in range(k)]
    q_keys = [simple_ot(seeds0[j], seeds1[j], int(s[j])) for j in range(k)]

    # ── Step 3: R builds T (k×m) and correction U ─────────────────────────
    T = np.array([prg(seeds0[j], m) for j in range(k)], dtype=np.uint8)   # (k, m)
    U = np.array(
        [prg(seeds1[j], m) ^ T[j] ^ r for j in range(k)], dtype=np.uint8
    )                                                                       # (k, m)

    # ── Step 4: S builds Q (k×m) ─────────────────────────────────────────
    Q = np.array(
        [prg(q_keys[j], m) ^ (s[j] * U[j]) for j in range(k)], dtype=np.uint8
    )                                                                       # (k, m)

    # ── Step 5: Transfer m messages using hashing ─────────────────────────
    Qt = Q.T   # (m, k): Qt[i] = i-th row of Q  =  Q_i
    Tt = T.T   # (m, k): Tt[i] = i-th row of T  =  T_i

    results = []
    for i in range(m):
        key0 = H(i, Qt[i])
        key1 = H(i, (Qt[i] ^ s).astype(np.uint8))

        e0 = xor_bytes(sender_m0[i], key0[:len(sender_m0[i])])
        e1 = xor_bytes(sender_m1[i], key1[:len(sender_m1[i])])

        recv_key = H(i, Tt[i])
        chosen   = e0 if r[i] == 0 else e1
        results.append(xor_bytes(chosen, recv_key[:len(sender_m0[i])]))

    return results


# ── correctness check (k=16 for speed) ───────────────────────────────────
_m0 = [b"msg_0_" + bytes([i]) for i in range(20)]
_m1 = [b"msg_1_" + bytes([i]) for i in range(20)]
_ch = np.array([i % 2 for i in range(20)], dtype=np.uint8)
_res = iknp_ote(_m0, _m1, _ch, k=16)
for i in range(20):
    assert _res[i] == (_m0[i] if _ch[i] == 0 else _m1[i])
print("IKNP OTE   ✓  correctness check passed")


def bench_iknp_ote(n: int, k: int = 128) -> float:
    """Run IKNP OTE for n messages. Returns total wall-clock time in ms."""
    m0 = [secrets.token_bytes(16)] * n
    m1 = [secrets.token_bytes(16)] * n
    choices = np.array([i % 2 for i in range(n)], dtype=np.uint8)
    t0 = time.perf_counter()
    iknp_ote(m0, m1, choices, k=k)
    return (time.perf_counter() - t0) * 1000


In [ ]:
# Plot the graphs

# Extended n range — goes well past the IKNP crossover point (~k=128)
N_VALUES = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]

# Number of repetitions to average over — smooths out OS/cache noise
REPS = 5


def bench_avg(bench_fn, n, reps=REPS, **kwargs) -> tuple[float, float]:
    """
    Run bench_fn(n) `reps` times and return (mean_ms, std_ms).
    Averaging eliminates the OS scheduling noise that caused the
    IKNP curve to look jagged at small n.
    """
    times = [bench_fn(n, **kwargs) for _ in range(reps)]
    return float(np.mean(times)), float(np.std(times))


print(f"\nBenchmarking Simple OT  (averaged over {REPS} runs)…")
simple_mean, simple_std = [], []
for n in N_VALUES:
    mean, std = bench_avg(bench_simple_ot, n)
    simple_mean.append(mean)
    simple_std.append(std)
    print(f"  n = {n:5d}   {mean:9.2f} ± {std:.2f} ms")

print(f"\nBenchmarking IKNP OTE  k=128  (averaged over {REPS} runs)…")
iknp_mean, iknp_std = [], []
for n in N_VALUES:
    mean, std = bench_avg(bench_iknp_ote, n, k=128)
    iknp_mean.append(mean)
    iknp_std.append(std)
    print(f"  n = {n:5d}   {mean:9.2f} ± {std:.2f} ms")

# ── amortised per-OT cost ─────────────────────────────────────────────────
simple_per_ot = [m / n for m, n in zip(simple_mean, N_VALUES)]
iknp_per_ot   = [m / n for m, n in zip(iknp_mean,  N_VALUES)]

# ── plot ──────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"Simple OT vs IKNP OT Extension  (averaged over {REPS} runs)",
    fontsize=14, fontweight='bold'
)

# Left — total time with shaded ±1 std band
ax1.plot(N_VALUES, simple_mean, 'o-', color='crimson',   lw=2, label='Simple OT')
ax1.fill_between(N_VALUES,
                 [m - s for m, s in zip(simple_mean, simple_std)],
                 [m + s for m, s in zip(simple_mean, simple_std)],
                 color='crimson', alpha=0.15)

ax1.plot(N_VALUES, iknp_mean, 's-', color='steelblue', lw=2, label='IKNP OTE')
ax1.fill_between(N_VALUES,
                 [m - s for m, s in zip(iknp_mean, iknp_std)],
                 [m + s for m, s in zip(iknp_mean, iknp_std)],
                 color='steelblue', alpha=0.15)

ax1.set_xscale('log', base=2)
ax1.set_xlabel('Number of OTs (n)', fontsize=12)
ax1.set_ylabel('Total Time (ms)',   fontsize=12)
ax1.set_title('Total Execution Time')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Right — amortised cost per OT
ax2.plot(N_VALUES, simple_per_ot, 'o-', color='crimson',
         lw=2, label='Simple OT  (constant per-OT cost)')
ax2.plot(N_VALUES, iknp_per_ot,   's-', color='steelblue',
         lw=2, label='IKNP OTE  (amortised cost drops)')

# Mark the crossover point
crossover_n = None
for i in range(len(N_VALUES) - 1):
    if iknp_per_ot[i] > simple_per_ot[i] and iknp_per_ot[i+1] <= simple_per_ot[i+1]:
        crossover_n = N_VALUES[i+1]
        ax2.axvline(crossover_n, color='gray', lw=1.5, ls='--', alpha=0.7,
                    label=f'Crossover ≈ n={crossover_n}')
        break

ax2.set_xscale('log', base=2)
ax2.set_xlabel('Number of OTs (n)', fontsize=12)
ax2.set_ylabel('Time per OT (ms)',   fontsize=12)
ax2.set_title('Amortised Cost per OT')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ot_vs_ote.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'═'*50}")
print(f"  Summary at n = {N_VALUES[-1]} OTs")
print(f"{'═'*50}")
print(f"  Simple OT :  {simple_per_ot[-1]:.4f} ms / OT")
print(f"  IKNP OTE  :  {iknp_per_ot[-1]:.4f} ms / OT")
print(f"  Speedup   :  {simple_per_ot[-1] / iknp_per_ot[-1]:.1f}×")
if crossover_n:
    print(f"  Crossover :  n ≈ {crossover_n}  (IKNP wins beyond this point)")